## LangChain Chains 

In [ ]:
#!pip install pandas

In [35]:
import pandas as pd
df = pd.read_csv('data/reviews.csv')

df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLM Chain

## Runnable-based LangChain API
## (LCEL-LangChain Expression Language)


In [ ]:
""" 
DEPRECATED: The following code is deprecated and will be removed in future versions.

from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(temperature=0.9, model=llm_model)

prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

chain = LLMChain(llm=llm, prompt=prompt)

product = "Queen Size Sheet Set"
chain.run(product) """

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_model = "gpt-4o-mini"

llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [8]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe "
    "a company that makes {product}?"
)

In [9]:
# LCEL chain: prompt -> llm -> parser, composed with the pipe operator
chain = prompt | llm | StrOutputParser()

In [10]:
product = "Queen Size Sheet Set"
result = chain.invoke({"product": product})
print(result)

Choosing the right name for a company that specializes in queen size sheet sets can help convey the essence of your brand and attract customers. Here are some suggestions:

1. **Queenly Comfort**
2. **Royal Rest Sheets**
3. **Sheet Haven**
4. **Majestic Bedding**
5. **Queen Sheet Co.**
6. **Serene Queen Sheets**
7. **Regal Bedding**
8. **Crown Comfort Sheets**
9. **Dreamy Queen Sets**
10. **LuxQueen Linens**

Consider what values or qualities you want to emphasize—such as comfort, luxury, or style—and choose a name that reflects that vision. Additionally, check for domain availability if you plan to create an online presence.


## Simple Sequential Chain

In [ ]:
""" 
DEPRECATED: The following code is deprecated and will be removed in future versions.

from langchain.chains import SimpleSequentialChain

llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

overall_simple_chain.run(product) """

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [18]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [19]:
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe "
    "a company that makes {product}?"
)

In [20]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following "
    "company:{company_name}"
)

In [22]:
# chain 1: product -> company name (string)
chain_one = first_prompt | llm | StrOutputParser()

In [23]:
# chain 2: company_name -> description (string)
chain_two = second_prompt | llm | StrOutputParser()

In [24]:
# overall chain: chain_one's output becomes chain_two's "company_name" input
overall_simple_chain = (
    {"company_name": chain_one}
    | chain_two
)

In [25]:

product = "Queen Size Sheet Set"
result = overall_simple_chain.invoke({"product": product})
print(result)

Discover luxurious comfort with our queen size sheet sets, designed for style and quality to elevate your sleeping experience.


## Sequential Chain

In [ ]:
""" 
DEPRECATED: The following code is deprecated and will be removed in future versions.

from langchain.chains import SequentialChain

llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )

second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )


# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )

# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

review = df.Review[5]
overall_chain(review) """

In [27]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [28]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [29]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
chain_one = first_prompt | llm | StrOutputParser()

In [30]:
# prompt template 2: summarize
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
chain_two = second_prompt | llm | StrOutputParser()

In [31]:
# prompt template 3: detect language (branches off original Review)
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
chain_three = third_prompt | llm | StrOutputParser()

In [32]:
# prompt template 4: follow-up message (needs summary + language)
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_four = fourth_prompt | llm | StrOutputParser()

In [33]:
# Overall chain: input {"Review": ...} -> accumulates keys as it flows through
overall_chain = (
    RunnablePassthrough.assign(English_Review=chain_one)
    | RunnablePassthrough.assign(summary=chain_two)
    | RunnablePassthrough.assign(language=chain_three)
    | RunnablePassthrough.assign(followup_message=chain_four)
)

In [36]:
review = df.Review[5]
result = overall_chain.invoke({"Review": review})

In [37]:
print(result["English_Review"])
print(result["summary"])
print(result["language"])
print(result["followup_message"])

I find the taste mediocre. The foam doesn't hold, it's strange. I buy the same ones commercially and the taste is much better... Old batch or counterfeit!?
The reviewer finds the product's taste mediocre and the foam unsatisfactory, comparing it unfavorably to a better version they buy commercially, leading them to question its authenticity.
The review is written in French.
Merci pour votre retour détaillé. Nous apprécions vos commentaires sur le goût du produit et la qualité de la mousse. Votre expérience est importante pour nous, et nous sommes désolés d'apprendre que notre produit n'a pas répondu à vos attentes. Nous nous efforçons constamment d'améliorer nos recettes et de garantir l'authenticité de nos ingrédients. Pourriez-vous nous donner plus de détails sur ce que vous recherchez dans un produit similaire ? Votre retour nous aidera à nous améliorer et à mieux répondre aux besoins de nos clients. Merci encore pour votre franchise !


## Router Chain

<!-- from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0, model=llm_model)


destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)


MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""


router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

chain.run("What is black body radiation?")

chain.run("what is 2 + 2")

chain.run("Why does every cell in our body contain DNA?") -->

In [1]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [2]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch
import json
import re

In [5]:
llm_model = "gpt-4o-mini"

llm = ChatOpenAI(temperature=0, model=llm_model)

In [6]:
# --- Build destination chains (same idea as before) ---
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    destination_chains[name] = prompt | llm | StrOutputParser()

In [7]:
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [8]:
default_chain = ChatPromptTemplate.from_template("{input}") | llm | StrOutputParser()

In [21]:
# --- Router prompt (same template, still works as a plain PromptTemplate) ---
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
"destination": string \\ "DEFAULT" or name of the prompt to use in {destinations}
"next_inputs": string \\ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of "destination" MUST match one of \
the candidate prompts listed below.\
If "destination" does not fit any of the specified prompts, set it to "DEFAULT."
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [22]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)

In [23]:
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
)

In [24]:
# --- Custom parser to replace RouterOutputParser ---
def parse_router_output(text: str) -> dict:
    """Extract the ```json ... ``` block and parse it."""
    match = re.search(r"```json(.*?)```", text, re.DOTALL)
    json_str = match.group(1).strip() if match else text.strip()
    parsed = json.loads(json_str)
    return {
        "destination": parsed["destination"],
        "next_inputs": {"input": parsed["next_inputs"]},
    }

In [25]:
router_chain = router_prompt | llm | StrOutputParser() | RunnableLambda(parse_router_output)

In [26]:
# --- Routing function: dispatch to the right destination chain ---
def route(route_info: dict):
    destination = route_info["destination"]
    next_inputs = route_info["next_inputs"]
    chosen_chain = destination_chains.get(destination, default_chain)
    return chosen_chain.invoke(next_inputs)


In [27]:
chain = router_chain | RunnableLambda(route)

In [30]:
# --- Usage ---
print(chain.invoke({"input": "What is black body radiation?"}))
print(chain.invoke({"input": "what is 2 + 2"}))
print(chain.invoke({"input": "Why does every cell in our body contain DNA?"}))

Black body radiation refers to the electromagnetic radiation emitted by an idealized object known as a "black body." A black body is a perfect absorber of all incident radiation, meaning it absorbs all wavelengths of light and does not reflect any. When a black body is heated, it emits radiation across a continuous spectrum of wavelengths.

The key characteristics of black body radiation include:

1. **Temperature Dependence**: The amount and spectrum of radiation emitted depend solely on the temperature of the black body. As the temperature increases, the total amount of emitted radiation increases, and the peak wavelength of the emitted radiation shifts to shorter wavelengths (this is described by Wien's displacement law).

2. **Planck's Law**: The spectral distribution of black body radiation is described by Planck's law, which quantifies the intensity of radiation emitted at different wavelengths for a given temperature. This law was pivotal in the development of quantum mechanics.